# 健行筆記軌跡數據解析範例

本單元介紹如何使用 `BeautifulSoup` 解析「健行筆記」軌跡詳細頁面（`response.text`）中的運動指標數據（如日期、花費時間、爬升高度、下降高度、里程等）。

In [ ]:
from pprint import pprint
import requests
import pprint
from bs4 import BeautifulSoup as bs4
import ast
import re

cookies = {
    '_qg_fts': '1767931746',
    'QGUserId': '7370895265489497',
    'airisTracker': 'PBVn1p7lgF8c',
    '_cc_id': '4e8b3cae7d543f4a9d88a12f901bff5c',
    '__htid': 'ce1574f4-cb2e-4daf-bade-c85805e72fae',
    'AviviD_uuid': 'a8427030-bdb8-4a6d-a4e0-ea6d251285f2',
    'webuserid': '074de9c8-faff-0f5b-702a-9f3fb714febf',
    'AviviD_refresh_uuid_status': '2',
    'cookieConsent': 'true',
    'cookieConsent': 'true',
    '_ga_M7E3P87KRC': 'GS2.1.s1768464677$o3$g0$t1768464677$j60$l0$h1043739287',
    'aiq_cs_5a937136420cfdf368a8': '[%22https:%22%2C%22biji.co%22%2C%22hiking%22%2C[[%22hiking%22%2C%22g%22]]]',
    'cto_bundle': 'RedD_V82ZDl0M01BWmElMkZTMVolMkJsVFllaXFEbk5CbGh5T25rMjJ2NEppWTZHbkVmcWdMM1ZWTzZ6WmxyckVUV0p1UXJoZG1DcFJsbGh2SmZjMldwMXQ1OXpNaTZlSFRuJTJCOCUyQkY1eEltTGtpalBqaFFxdjZuWiUyRlFqV1RTZkN6QUZWRk1Wa2VVMWJyMkJPS01lMmY0dnprU2hKQUtRJTNEJTNE',
    'jiyakeji_uuid': '169f10e0-f6a7-11f0-a3f1-55d1c6e8be76',
    '_pubcid': 'f71f7e42-b01e-45a2-a05d-6ebade1562b6',
    '_fbp': 'fb.1.1782965731574.252662787824864391',
    'panoramaId_expiry': '1783570531614',
    'panoramaId': 'a625a5ce4e2b30bf5f1e47ad705c16d53938f2907b2a6422fb89c4f21f88eee8',
    'panoramaIdType': 'panoIndiv',
    '_gid': 'GA1.2.1815987775.1782965732',
    '_qg_cm': '2',
    '_ht_hi': '1',
    'pacid': 'pp.1.287506569.1780649264125',
    'AviviD_loadscript_log': '1',
    '_gcl_au': '1.1.1683444823.1782965734',
    '_ss_pp_id': 'pp.1.660576542.1769047827201',
    'AviviD_already_exist': '1',
    '_ht_47b240': '1',
    'show_avivid_native_subscribe': '2',
    '_ht_f3244e': '1',
    'AviviD_session_id': '1782979270864',
    '_ga': 'GA1.1.1068798473.1767931747',
    'FCCDCF': '%5Bnull%2Cnull%2Cnull%2Cnull%2Cnull%2Cnull%2C%5B%5B32%2C%22%5B%5C%22cbf7ea62-ac9c-4528-a944-f06e4960a97e%5C%22%2C%5B1767931746%2C578000000%5D%5D%22%5D%5D%5D',
    'cf_clearance': 'HgQPR3VdBHrHNAtp6xecjoLmR6LF6mqdx6KmfAfYm1M-1782979273-1.2.1.1-Sb9JsC3PHkA1_HwJw1OiYvEuMeqNhoipUshb.TdKycNLwkHQ4qN5MTx2ojIONyCAhYyxKSBIFYIlYzAMb5oP1l51BkQdvK3OdhzlULBYLFYCAfQACVyrfYUp64GkxgamkjlqcwTtgGoizgUy3h3XpnpKpJ7UFBY68TfzD_SH6aQCNLwDZz0Cvuxq6PBgnH7KuP4dE5_bqPOFkrpsgrOdwPkABY5KyybtL8qugONHTvF21v0q4pEkYtxwmAAERAkQIkYd6RT3H979Lis6xf92V.zqWsqKPipk3aVax1aZI25dt2KpFdAR82NJ4K..O0rkXTI3V4hO_tMvnQuzCeIs8A',
    '_qg_pushrequest': 'true',
    'FCNEC': '%5B%5B%22AKsRol8YwJY1R9ETGatn0cVv0bvdWZo4ymG8E4kW5AjoqXyKqZ4Qa95DkdhL1E4BzHXgYOtrgBIAjURoMKKwIKGQuyRk05XJ1RjhFig0_z2_1nVbo-3ijq2NEPaVXxx9rII4O6NHNUpSsD5vKoXELabxEqOODwLCCA%3D%3D%22%5D%5D',
    '_td': '7fc0fa11-e78e-4b6d-81a2-24ca84474bd6',
    '__eoi': 'ID=b591f2634a33386f:T=1767931748:RT=1782979278:S=AA-AfjbTDG5cRq_LBETDLFXt9xdJ',
    '__gads': 'ID=e8a48324d2edb2fd:T=1767931748:RT=1782979278:S=ALNI_MbfI3KSF1mjQtEovNOJXNPN3b609g',
    '__gpi': 'UID=000011dfe7676648:T=1767931748:RT=1782979278:S=ALNI_MYM8VuVDx-kWA_8AR5i6JHF_UzK7g',
    '_ga_B7QHK7HLYB': 'GS2.1.s1782979245$o5$g1$t1782979278$j27$l0$h0',
}

headers = {
    'accept': '*/*',
    'accept-language': 'zh-TW,zh;q=0.9,en-US;q=0.8,en;q=0.7',
    'priority': 'u=1, i',
    'referer': 'https://hiking.biji.co/index.php?q=trail&act=detail&id=288',
    'sec-ch-ua': '"Google Chrome";v="149", "Chromium";v="149", "Not)A;Brand";v="24"',
    'sec-ch-ua-mobile': '?0',
    'sec-ch-ua-platform': '"Windows"',
    'sec-fetch-dest': 'empty',
    'sec-fetch-mode': 'cors',
    'sec-fetch-site': 'same-origin',
    'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/149.0.0.0 Safari/537.36',
    # 'cookie': '_qg_fts=1767931746; QGUserId=7370895265489497; airisTracker=PBVn1p7lgF8c; _cc_id=4e8b3cae7d543f4a9d88a12f901bff5c; __htid=ce1574f4-cb2e-4daf-bade-c85805e72fae; AviviD_uuid=a8427030-bdb8-4a6d-a4e0-ea6d251285f2; webuserid=074de9c8-faff-0f5b-702a-9f3fb714febf; AviviD_refresh_uuid_status=2; cookieConsent=true; cookieConsent=true; _ga_M7E3P87KRC=GS2.1.s1768464677$o3$g0$t1768464677$j60$l0$h1043739287; aiq_cs_5a937136420cfdf368a8=[%22https:%22%2C%22biji.co%22%2C%22hiking%22%2C[[%22hiking%22%2C%22g%22]]]; cto_bundle=RedD_V82ZDl0M01BWmElMkZTMVolMkJsVFllaXFEbk5CbGh5T25rMjJ2NEppWTZHbkVmcWdMM1ZWTzZ6WmxyckVUV0p1UXJoZG1DcFJsbGh2SmZjMldwMXQ1OXpNaTZlSFRuJTJCOCUyQkY1eEltTGtpalBqaFFxdjZuWiUyRlFqV1RTZkN6QUZWRk1Wa2VVMWJyMkJPS01lMmY0dnprU2hKQUtRJTNEJTNE; jiyakeji_uuid=169f10e0-f6a7-11f0-a3f1-55d1c6e8be76; _pubcid=f71f7e42-b01e-45a2-a05d-6ebade1562b6; _fbp=fb.1.1782965731574.252662787824864391; panoramaId_expiry=1783570531614; panoramaId=a625a5ce4e2b30bf5f1e47ad705c16d53938f2907b2a6422fb89c4f21f88eee8; panoramaIdType=panoIndiv; _gid=GA1.2.1815987775.1782965732; _qg_cm=2; _ht_hi=1; pacid=pp.1.287506569.1780649264125; AviviD_loadscript_log=1; _gcl_au=1.1.1683444823.1782965734; _ss_pp_id=pp.1.660576542.1769047827201; AviviD_already_exist=1; _ht_47b240=1; show_avivid_native_subscribe=2; _ht_f3244e=1; AviviD_session_id=1782979270864; _ga=GA1.1.1068798473.1767931747; FCCDCF=%5Bnull%2Cnull%2Cnull%2Cnull%2Cnull%2Cnull%2C%5B%5B32%2C%22%5B%5C%22cbf7ea62-ac9c-4528-a944-f06e4960a97e%5C%22%2C%5B1767931746%2C578000000%5D%5D%22%5D%5D%5D; cf_clearance=HgQPR3VdBHrHNAtp6xecjoLmR6LF6mqdx6KmfAfYm1M-1782979273-1.2.1.1-Sb9JsC3PHkA1_HwJw1OiYvEuMeqNhoipUshb.TdKycNLwkHQ4qN5MTx2ojIONyCAhYyxKSBIFYIlYzAMb5oP1l51BkQdvK3OdhzlULBYLFYCAfQACVyrfYUp64GkxgamkjlqcwTtgGoizgUy3h3XpnpKpJ7UFBY68TfzD_SH6aQCNLwDZz0Cvuxq6PBgnH7KuP4dE5_bqPOFkrpsgrOdwPkABY5KyybtL8qugONHTvF21v0q4pEkYtxwmAAERAkQIkYd6RT3H979Lis6xf92V.zqWsqKPipk3aVax1aZI25dt2KpFdAR82NJ4K..O0rkXTI3V4hO_tMvnQuzCeIs8A; _qg_pushrequest=true; FCNEC=%5B%5B%22AKsRol8YwJY1R9ETGatn0cVv0bvdWZo4ymG8E4kW5AjoqXyKqZ4Qa95DkdhL1E4BzHXgYOtrgBIAjURoMKKwIKGQuyRk05XJ1RjhFig0_z2_1nVbo-3ijq2NEPaVXxx9rII4O6NHNUpSsD5vKoXELabxEqOODwLCCA%3D%3D%22%5D%5D; _td=7fc0fa11-e78e-4b6d-81a2-24ca84474bd6; __eoi=ID=b591f2634a33386f:T=1767931748:RT=1782979278:S=AA-AfjbTDG5cRq_LBETDLFXt9xdJ; __gads=ID=e8a48324d2edb2fd:T=1767931748:RT=1782979278:S=ALNI_MbfI3KSF1mjQtEovNOJXNPN3b609g; __gpi=UID=000011dfe7676648:T=1767931748:RT=1782979278:S=ALNI_MYM8VuVDx-kWA_8AR5i6JHF_UzK7g; _ga_B7QHK7HLYB=GS2.1.s1782979245$o5$g1$t1782979278$j27$l0$h0',
}

trail_data = {
    "tao_mountain": 429,
    "tao_kalaye": 1746,
    "chiyou_pintian": 1737,
    "mt_beidawu": 1750,
    "mt_taguan": 1761,
    "mt_hijiayan": 531,
    "mt_junda": 500,
    "mt_xue_east": 1734,
    "mt_guanshangling": 1760,
    "hehuan_north": 288,
    "hehuan_north_west": 536,
    "mt_jade_front": 68
}
for trail_name, hiking_note_id in trail_data.items():
    for page in range(1, 10):
        params = {
            'id': hiking_note_id,
            'page': page,
            'device': 'computer',
        }
        response = requests.get('https://hiking.biji.co/trail/ajax/load_related_gpx', params=params, cookies=cookies, headers=headers)
        html = response.json()["data"]["list"]
        soup = bs4(f"<ul>{html}</ul>", 'lxml')

        records =[]
        items = soup.ul.find_all("li", class_="flex", recursive=False)


        for record in items:
            # raw_record = {
            #     "distance": record.select("span")[0].text,
            #     "duration": record.select("span.truncate")[0].text,
            #     "ascent": record.select("span")[2].text,
            #     "descent": record.select("span")[3].text,
            #     "record_date": record.select("span")[4].text,
            # }

            # distance_km (里程)
            distance = record.select('span')[0].text
            distance_km = re.search(r"([\d.]+)", distance).group(1)

            # duration_minutes (耗時)
            duration = record.select('span.truncate')[0].text.strip()
            hour_match = re.search(r"(\d+)\s*小時", duration)
            minute_match = re.search(r"(\d+)\s*分鐘", duration)

            hours = int(hour_match.group(1)) if hour_match else 0
            minutes = int(minute_match.group(1)) if minute_match else 0

            duration_minutes = hours * 60 +minutes

            # ascent_m (爬升)
            ascent = record.select('span')[2].text.replace(",", "").strip()
            ascent_m = re.search(r"(\d+)", ascent).group(1)

            # descent_m (下降)
            descent = record.select('span')[3].text.replace(",", "").strip()
            descent_m = re.search(r"(\d+)", descent).group(1)

            # record_date
            record_date = record.select('span')[4].text

            clean_record = {
                "trail_name" :trail_name,
                "distance_km" : distance_km,
                "duration_minutes" : duration_minutes,
                "ascent_m" : ascent_m,
                "descent_m" : descent_m,
                "record_date" : record_date,
            }
            records.append(clean_record)
        pprint.pprint(records)

[{'ascent_m': '1517',
  'descent_m': '1435',
  'distance_km': '11.26',
  'duration_minutes': 584,
  'record_date': '2026-06-20',
  'trail_name': 'tao_mount'},
 {'ascent_m': '1456',
  'descent_m': '69',
  'distance_km': '5.46',
  'duration_minutes': 470,
  'record_date': '2026-06-17',
  'trail_name': 'tao_mount'},
 {'ascent_m': '1682',
  'descent_m': '1674',
  'distance_km': '17.90',
  'duration_minutes': 851,
  'record_date': '2026-05-31',
  'trail_name': 'tao_mount'},
 {'ascent_m': '2785',
  'descent_m': '2779',
  'distance_km': '18.52',
  'duration_minutes': 680,
  'record_date': '2026-05-31',
  'trail_name': 'tao_mount'},
 {'ascent_m': '1499',
  'descent_m': '1498',
  'distance_km': '10.64',
  'duration_minutes': 598,
  'record_date': '2026-05-24',
  'trail_name': 'tao_mount'},
 {'ascent_m': '1433',
  'descent_m': '1581',
  'distance_km': '12.53',
  'duration_minutes': 563,
  'record_date': '2026-05-24',
  'trail_name': 'tao_mount'},
 {'ascent_m': '1368',
  'descent_m': '1352',
  'd

In [ ]:


html = response.json()["data"]["list"]
soup = bs4(f"<ul>{html}</ul>", 'lxml')

records =[]
items = soup.ul.find_all("li", class_="flex", recursive=False)


for record in items:
    # raw_record = {
    #     "distance": record.select("span")[0].text,
    #     "duration": record.select("span.truncate")[0].text,
    #     "ascent": record.select("span")[2].text,
    #     "descent": record.select("span")[3].text,
    #     "record_date": record.select("span")[4].text,
    # }

    # distance_km (里程)
    distance = record.select('span')[0].text
    distance_km = re.search(r"([\d.]+)", distance).group(1)

    # duration_minutes (耗時)
    duration = record.select('span.truncate')[0].text.strip()
    hour_match = re.search(r"(\d+)\s*小時", duration)
    minute_match = re.search(r"(\d+)\s*分鐘", duration)

    hours = int(hour_match.group(1)) if hour_match else 0
    minutes = int(minute_match.group(1)) if minute_match else 0

    duration_minutes = hours * 60 +minutes

    # ascent_m (爬升)
    ascent = record.select('span')[2].text.strip()
    ascent_m = re.search(r"(\d+)", ascent).group(1)

    # descent_m (下降)
    descent = record.select('span')[3].text.strip()
    descent_m = re.search(r"(\d+)", descent).group(1)


    # record_date
    record_date = record.select('span')[4].text

    clean_record = {
        "distance_km" : distance_km,
        "duration_minutes" : duration_minutes,
        "ascent_m" : ascent_m,
        "descent_m" : descent_m,
        "record_date" : record_date,
    }
    records.append(clean_record)
pprint.pprint(records)





[{'ascent_m': '476',
  'descent_m': '494',
  'distance_km': '4.90',
  'duration_minutes': 182,
  'record_date': '2020-12-26'},
 {'ascent_m': '476',
  'descent_m': '512',
  'distance_km': '4.80',
  'duration_minutes': 229,
  'record_date': '2026-06-28'},
 {'ascent_m': '529',
  'descent_m': '530',
  'distance_km': '5.42',
  'duration_minutes': 175,
  'record_date': '2026-06-28'},
 {'ascent_m': '490',
  'descent_m': '476',
  'distance_km': '4.50',
  'duration_minutes': 160,
  'record_date': '2026-06-23'},
 {'ascent_m': '531',
  'descent_m': '523',
  'distance_km': '5.31',
  'duration_minutes': 223,
  'record_date': '2026-06-23'},
 {'ascent_m': '1',
  'descent_m': '1',
  'distance_km': '14.43',
  'duration_minutes': 626,
  'record_date': '2026-06-20'},
 {'ascent_m': '1',
  'descent_m': '1',
  'distance_km': '14.62',
  'duration_minutes': 640,
  'record_date': '2026-06-20'},
 {'ascent_m': '480',
  'descent_m': '471',
  'distance_km': '4.41',
  'duration_minutes': 243,
  'record_date': '2026